In [ ]:
import easyocr
import cv2
import math
from ultralytics import YOLO 

# Definiciones fuera del bucle (¡esto está bien!)
model = YOLO("./runs/detect/train/weights/best.pt")
classes = {0:"licenseplate",1:"car"}
reader = easyocr.Reader(['en'], gpu=True) 
capture_video = cv2.VideoCapture("video5.mp4")

while(True):
    ret,frame_video = capture_video.read()
    if ret:
        results = model(frame_video,stream=True)
        
        for frames in results:
            boxes = frames.boxes
            for box in boxes:
                # Clase
                cls = int(box.cls[0])
                
                if cls not in classes.keys():
                    continue
                else:
                    # 1. Definir COORDENADAS aquí, para CUALQUIER objeto válido (car o licenseplate)
                    x1, y1, x2, y2 = box.xyxy[0]
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values

                    # Confianza
                    confidence = math.ceil((box.conf[0]*100))/100
                    print("Clase -->",classes[cls])
                    print("Confianza --->",confidence)

                    # 2. CALCULAR COLORES aquí, para CUALQUIER objeto válido
                    escala = int((cls / len(classes)) * 255 * 3)
                    # Usando un if/elif/else más sencillo para claridad
                    if escala >= 255 * 2:
                        R, G, B = 255, 255, escala - 255 * 2
                    elif escala >= 255:
                        R, G, B = 255, escala - 255, 0
                    else:
                        R, G, B = escala, 0, 0
                    
                    
                    # 3. Aplicar OCR SOLO si es una MATRÍCULA
                    if classes[cls] == "licenseplate":
                        license_plate_img = frame_video[y1:y2, x1:x2]
                        result_ocr = reader.readtext(
                            license_plate_img, 
                            allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                            detail=0
                        )
                        
                        plate_text=""
                        if result_ocr:
                            plate_text = "".join(result_ocr).replace(" ", "") 
                            print(f"Matrícula Detectada: {plate_text}")
                            
                            # Mostrar el texto del OCR
                            cv2.putText(
                                frame_video, 
                                plate_text, 
                                (x1, y1 - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 
                                1, 
                                (0, 255, 0), # Color verde para el texto OCR
                                2
                            )
                            print(f"Matrícula Detectada: {plate_text}")
                    
                    # 4. Dibujar el contenedor y clase (Esto ahora tiene acceso a x1, y1, x2, y2, R, G, B)
                    cv2.rectangle(frame_video, (x1, y1), (x2, y2), (R, G, B), 3)
                    
                    # Si no es matrícula, dibuja solo la clase; si es matrícula, se puede añadir el texto OCR.
                    # Aquí dibujamos el nombre de la clase, que es lo que estaba antes
                    cv2.putText(frame_video, classes[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)
                    
            cv2.imshow('capture_video', frame_video)
        
        if cv2.waitKey(20) == 27:
            break
            
# Libera el objeto de captura
capture_video.release()
# Destruye ventanas
cv2.destroyAllWindows()

In [ ]:
import pytesseract
from pytesseract import Output
import cv2
import math
from ultralytics import YOLO 


# Asegúrate de instalar Tesseract OCR en tu sistema
# y ajusta la ruta a tu instalación. (Ruta de ejemplo para Windows)
# Si estás en Linux/macOS y lo instalaste con el gestor de paquetes, puede que no necesites esta línea.
tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract'
pytesseract.pytesseract.tesseract_cmd = tesseract_cmd 

# Definiciones fuera del bucle
model = YOLO("./runs/detect/train/weights/best.pt")
classes = {0:"licenseplate",1:"car"}

capture_video = cv2.VideoCapture("video5.mp4")

while(True):
    ret,frame_video = capture_video.read()
    if ret:
        results = model(frame_video,stream=True)
        
        for frames in results:
            boxes = frames.boxes
            for box in boxes:
                # Clase
                cls = int(box.cls[0])
                
                if cls not in classes.keys():
                    continue
                else:
                    # 1. Definir COORDENADAS aquí
                    x1, y1, x2, y2 = box.xyxy[0]
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) 

                    # Confianza
                    confidence = math.ceil((box.conf[0]*100))/100
                    print("Clase -->",classes[cls])
                    print("Confianza --->",confidence)

                    # 2. CALCULAR COLORES aquí
                    escala = int((cls / len(classes)) * 255 * 3)
                    if escala >= 255 * 2:
                        R, G, B = 255, 255, escala - 255 * 2
                    elif escala >= 255:
                        R, G, B = 255, escala - 255, 0
                    else:
                        R, G, B = escala, 0, 0
                    
                    
                    # 3. Aplicar OCR SOLO si es una MATRÍCULA
                    if classes[cls] == "licenseplate":
                        license_plate_img = frame_video[y1:y2, x1:x2]
                        
                       
                        ocr_config = r'--psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
                        
                        try:
                            plate_text = pytesseract.image_to_string(
                                license_plate_img, 
                                config=ocr_config
                            )
                            # Post-procesamiento: Limpiar el texto de saltos de línea y espacios
                            plate_text = plate_text.strip().replace(" ", "").replace("\n", "")
                        except pytesseract.TesseractNotFoundError:
                            plate_text = "Tesseract NO instalado!"
                        except Exception as e:
                            plate_text = f"Error OCR: {e}"

                        
                        if plate_text:
                            print(f"Matrícula Detectada: {plate_text}")
                            
                            # Mostrar el texto del OCR
                            cv2.putText(
                                frame_video, 
                                plate_text, 
                                (x1, y1 - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 
                                1, 
                                (0, 255, 0), # Color verde
                                2
                            )
                    
                    # 4. Dibujar el contenedor y clase
                    cv2.rectangle(frame_video, (x1, y1), (x2, y2), (R, G, B), 3)
                    cv2.putText(frame_video, classes[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)
                    
            cv2.imshow('capture_video', frame_video)
        
        if cv2.waitKey(20) == 27:
            break
            
# Libera el objeto de captura
capture_video.release()
# Destruye ventanas
cv2.destroyAllWindows()

In [ ]:
import paddleocr
import cv2
import math
from ultralytics import YOLO 
import logging 
import re 
import numpy as np

# --- 1. CONFIGURACIÓN INICIAL Y SILENCIO DE LOGS ---
logging.getLogger('ppocr').setLevel(logging.ERROR) 
logging.getLogger('paddle').setLevel(logging.ERROR)
# ----------------------------------------------------

# --- 2. INICIALIZACIÓN DE MODELOS ---
try:
    model = YOLO("./runs/detect/train/weights/best.pt")
    classes = {0:"licenseplate",1:"car"}
    yolo_status = "OK"
    print("✅ YOLO inicializado correctamente")
except Exception as e:
    print(f"❌ ERROR FATAL: No se pudo cargar YOLO/PyTorch: {e}")
    yolo_status = "FALLO"
    
# INICIALIZACIÓN PADDLEOCR
ocr_status = "FALLO"
ocr = None

try:
    print("🔄 Intentando inicializar PaddleOCR...")
    ocr = paddleocr.PaddleOCR(lang='en', use_angle_cls=False)
    ocr_status = "OK"
    print("✅ PaddleOCR inicializado correctamente")
except Exception as e:
    print(f"❌ Error inicializando PaddleOCR: {e}")
    ocr_status = "FALLO"

capture_video = cv2.VideoCapture("Video.mp4")

if not capture_video.isOpened():
    print("❌ No se pudo abrir el video")
    exit()

print("🎥 Video abierto correctamente")

# --- 3. BUCLE PRINCIPAL DE PROCESAMIENTO DE VIDEO ---
frame_count = 0
processed_plates = 0

def extract_text_from_ocr(result_ocr):
    """Extrae texto CORRECTAMENTE de los resultados de PaddleOCR"""
    texts = []
    
    if result_ocr is None or not result_ocr:
        return texts
        
    try:
        # La estructura correcta es: [{'rec_texts': ['TEXTO1', 'TEXTO2'], ...}]
        for page_result in result_ocr:
            if isinstance(page_result, dict) and 'rec_texts' in page_result:
                # Extraer los textos reconocidos directamente
                rec_texts = page_result['rec_texts']
                if rec_texts:
                    texts.extend(rec_texts)
                    
                    # DEBUG: Mostrar confianza también
                    if 'rec_scores' in page_result:
                        scores = page_result['rec_scores']
                        print(f"Texto detectado: {rec_texts} con confianzas: {scores}")
            
    except Exception as e:
        print(f"Error extrayendo texto: {e}")
    return texts

while(True):
    ret, frame_video = capture_video.read()
    if not ret:
        print("⏹️ Fin del video")
        break
    
    frame_count += 1
    
    if yolo_status == "OK":
        results = model(frame_video, stream=True)
    else:
        results = [] 
        cv2.putText(frame_video, "YOLO OFFLINE", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    
    for frames in results:
        boxes = frames.boxes
        for box in boxes:
            cls = int(box.cls[0])
            
            if cls not in classes.keys():
                continue
            
            # 1. Definir COORDENADAS
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) 

            confidence = math.ceil((box.conf[0]*100))/100
            plate_text = ""

            # 3. Aplicar OCR SOLO si es una MATRÍCULA
            if classes[cls] == "licenseplate":
                
                license_plate_img = frame_video[y1:y2, x1:x2]

                # VALIDACIÓN: El recorte debe ser válido
                if (license_plate_img is None or license_plate_img.size == 0 or 
                    license_plate_img.shape[0] <= 10 or license_plate_img.shape[1] <= 10):
                    plate_text = "INVALID CROP"
                    continue

                if ocr_status == "OK" and ocr is not None:
                    try:
                        # DEBUG para primeras detecciones
                        debug_mode = processed_plates < 5
                        
                        if debug_mode:
                            print(f"\n🎯 FRAME {frame_count} - PROCESANDO MATRÍCULA")
                            print(f"Tamaño: {license_plate_img.shape}")
                        
                        # USAR predict()
                        result = ocr.predict(license_plate_img)
                        
                        # Extraer textos CORRECTAMENTE
                        detected_texts = extract_text_from_ocr(result)
                        
                        if debug_mode:
                            print(f"Textos detectados: {detected_texts}")
                        
                        # Combinar todos los textos
                        combined_text = "".join(detected_texts)
                        
                        # Limpieza
                        if combined_text:
                            cleaned_text = combined_text.replace(" ", "").replace("\n", "").upper()
                            plate_text = re.sub(r'[^A-Z0-9]', '', cleaned_text)
                            
                            if 3 <= len(plate_text) <= 12:
                                processed_plates += 1
                                if debug_mode:
                                    print(f"✅ MATRÍCULA DETECTADA: '{plate_text}'")
                            else:
                                plate_text = "INVALID LENGTH"
                                if debug_mode:
                                    print(f"❌ Longitud inválida: '{plate_text}' ({len(plate_text)} caracteres)")
                        else:
                            plate_text = "NO TEXT"
                            if debug_mode:
                                print("❌ No se detectó texto")

                    except Exception as e:
                        print(f"❌ Error en OCR (Frame {frame_count}): {str(e)}")
                        plate_text = "OCR ERROR"
                else:
                    plate_text = "OCR OFFLINE"
            
            # 4. Visualización
            color = (0, 255, 0) if classes[cls] == "car" else (255, 0, 0)
            cv2.rectangle(frame_video, (x1, y1), (x2, y2), color, 2)
            
            # Texto a mostrar
            if classes[cls] == "licenseplate":
                if plate_text and plate_text not in ["NO TEXT", "INVALID CROP", "INVALID LENGTH", "OCR OFFLINE", "OCR ERROR"]:
                    display_text = plate_text
                    text_color = (0, 255, 0)  # Verde
                else:
                    display_text = f"Plate: {plate_text}"
                    text_color = (0, 0, 255)  # Rojo
            else:
                display_text = f"{classes[cls]} {confidence:.2f}"
                text_color = (255, 255, 255)  # Blanco
            
            cv2.putText(frame_video, display_text, (x1, y1 - 10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2)

    # Mostrar estado en pantalla
    status_color = (0, 255, 0) if ocr_status == "OK" else (0, 0, 255)
    cv2.putText(frame_video, f"OCR: {ocr_status}", (10, 30), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
    cv2.putText(frame_video, f"Frame: {frame_count}", (10, 60), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(frame_video, f"Plates: {processed_plates}", (10, 80), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    cv2.imshow('capture_video', frame_video)
    if cv2.waitKey(20) == 27:
        print("⏹️ Video interrumpido por usuario")
        break

# Liberar recursos
capture_video.release()
cv2.destroyAllWindows()
print(f"🧹 Recursos liberados - Total matrículas procesadas: {processed_plates}")